## test_jsearch

In [1]:
import os
import json
import logging
from datetime import datetime, timezone
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

print("Imports OK")
print(f"RAPIDAPI_KEY loaded: {'yes' if os.getenv('RAPIDAPI_KEY') else '⚠️  NOT FOUND — check your .env file'}")

Imports OK
RAPIDAPI_KEY loaded: yes


In [2]:
JSEARCH_BASE_URL = "https://jsearch.p.rapidapi.com/search-v2"
RAPIDAPI_KEY     = os.getenv("RAPIDAPI_KEY")
RAPIDAPI_HOST    = "jsearch.p.rapidapi.com"

SEARCH_QUERIES = [
    "Data Analyst in New York",
    "Analytics Engineer in New York",
]

EXPLORATION_QUERY = SEARCH_QUERIES[0]  # single query for the live call
NUM_PAGES         = "1"                # 1 page = 1 API request consumed

CACHE_DIR  = Path("./cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / "jsearch_raw_response.json"

print(f"Exploration query : {EXPLORATION_QUERY!r}")
print(f"Cache path        : {CACHE_FILE.resolve()}")

Exploration query : 'Data Analyst in New York'
Cache path        : /Users/vanbrantley/code/nyc-data-job-market-tracker/ingestion/notebooks/cache/jsearch_raw_response.json


In [3]:
if CACHE_FILE.exists():
    print(f"Cache file already exists at {CACHE_FILE} — skipping live call.")
    print("Load the cached data in Cell 4 instead.")
else:
    if not RAPIDAPI_KEY:
        raise EnvironmentError("RAPIDAPI_KEY is not set. Add it to your .env file and re-run Cell 1.")

    headers = {
        "x-rapidapi-key":  RAPIDAPI_KEY,
        "x-rapidapi-host": RAPIDAPI_HOST,
        "Content-Type":    "application/json",
    }

    params = {
        "query":       EXPLORATION_QUERY,
        "num_pages":   NUM_PAGES,
        "country":     "us",
        "date_posted": "all",
    }

    log.info("Firing live API request — 1 of 200 monthly credits consumed.")

    try:
        response = requests.get(
            JSEARCH_BASE_URL,
            headers=headers,
            params=params,
            timeout=30,
        )
        response.raise_for_status()

        raw_data = response.json()

        if raw_data.get("status") != "OK":
            raise ValueError(f"API returned non-OK status: {raw_data}")

        with CACHE_FILE.open("w") as f:
            json.dump(raw_data, f, indent=2)

        log.info(f"Response saved to {CACHE_FILE}")

        print("\n--- Rate limit headers ---")
        for h in ["x-ratelimit-requests-remaining", "x-ratelimit-requests-limit", "x-ratelimit-requests-reset"]:
            print(f"  {h}: {response.headers.get(h, 'not present')}")

    except requests.exceptions.HTTPError as e:
        log.error(f"HTTP error: {e.response.status_code} — {e.response.text}")
        raise
    except requests.exceptions.RequestException as e:
        log.error(f"Network/request error: {e}")
        raise

2026-05-31 12:36:48,020 [INFO] Firing live API request — 1 of 200 monthly credits consumed.
2026-05-31 12:37:00,759 [INFO] Response saved to cache/jsearch_raw_response.json



--- Rate limit headers ---
  x-ratelimit-requests-remaining: 197
  x-ratelimit-requests-limit: 200
  x-ratelimit-requests-reset: 1751558


In [4]:
if not CACHE_FILE.exists():
    raise FileNotFoundError(f"No cache file found at {CACHE_FILE}. Run Cell 3 first.")

with CACHE_FILE.open() as f:
    raw_data = json.load(f)

print(f"Loaded from: {CACHE_FILE}")
print(f"Top-level keys: {list(raw_data.keys())}")

Loaded from: cache/jsearch_raw_response.json
Top-level keys: ['status', 'request_id', 'parameters', 'data']


In [5]:
print(f"status     : {raw_data.get('status')}")
print(f"request_id : {raw_data.get('request_id')}")

jobs = raw_data.get("data", {}).get("jobs", [])
print(f"\nJobs returned in this page : {len(jobs)}")

other_keys = [k for k in raw_data if k not in ("status", "request_id", "data")]
print(f"Other top-level keys       : {other_keys or 'none'}")

status     : OK
request_id : a9687534-b6c6-45ea-bf9e-54c74030a765

Jobs returned in this page : 10
Other top-level keys       : ['parameters']


In [6]:
if not jobs:
    print("No jobs in response — check Cell 5 output.")
else:
    first_job = jobs[0]
    print(f"=== Job 0: {first_job.get('job_title', 'N/A')} @ {first_job.get('employer_name', 'N/A')} ===\n")
    for key, value in first_job.items():
        display_val = value
        if isinstance(value, str) and len(value) > 120:
            display_val = value[:120] + " [...]"
        print(f"  {key:<40} {repr(display_val)}")

=== Job 0: Data Quality Analyst – Enterprise Data @ Bloomberg ===

  job_id                                   'lLMZZm-wci-hYzrpAAAAAA=='
  job_title                                'Data Quality Analyst – Enterprise Data'
  employer_name                            'Bloomberg'
  employer_logo                            'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRfpRRw4oauhOGkoKGseKHkYUUC7_M-M8COW0Gd&s=0'
  employer_website                         'https://www.bloomberg.com'
  job_publisher                            'LinkedIn'
  job_employment_type                      'Full-time'
  job_employment_types                     ['FULLTIME']
  job_apply_link                           'https://www.linkedin.com/jobs/view/data-quality-analyst-%E2%80%93-enterprise-data-at-bloomberg-4418336053'
  job_apply_is_direct                      False
  apply_options                            [{'apply_link': 'https://www.linkedin.com/jobs/view/data-quality-analyst-%E2%80%93-enterprise-data-at-b

In [7]:
from collections import Counter

field_counts = Counter()
for job in jobs:
    field_counts.update(job.keys())

total = len(jobs)
print(f"Total jobs: {total}")
print(f"\n{'Field':<45} {'Present':>7}  {'% records':>10}")
print("-" * 66)
for field, count in sorted(field_counts.items(), key=lambda x: -x[1]):
    bar = "█" * int((count / total) * 20) if total else ""
    print(f"  {field:<43} {count:>7}  {count/total*100:>9.0f}%  {bar}")

Total jobs: 10

Field                                         Present   % records
------------------------------------------------------------------
  job_id                                           10        100%  ████████████████████
  job_title                                        10        100%  ████████████████████
  employer_name                                    10        100%  ████████████████████
  employer_logo                                    10        100%  ████████████████████
  employer_website                                 10        100%  ████████████████████
  job_publisher                                    10        100%  ████████████████████
  job_employment_type                              10        100%  ████████████████████
  job_employment_types                             10        100%  ████████████████████
  job_apply_link                                   10        100%  ████████████████████
  job_apply_is_direct                              10      

In [9]:
null_counts = Counter()
for job in jobs:
    for k, v in job.items():
        if v is None or v == [] or v == "":
            null_counts[k] += 1

print(f"{'Field':<45} {'Null/empty':>10}  {'% records':>10}")
print("-" * 70)
for field, count in sorted(null_counts.items(), key=lambda x: -x[1]):
    print(f"  {field:<43} {count:>10}  {count/total*100:>9.0f}%")

if not null_counts:
    print("No null or empty fields found in this sample.")

Field                                         Null/empty   % records
----------------------------------------------------------------------
  job_salary                                          10        100%
  job_onet_soc                                        10        100%
  job_onet_job_zone                                   10        100%
  employer_reviews                                    10        100%
  job_salary_string                                    9         90%
  job_min_salary                                       9         90%
  job_max_salary                                       9         90%
  job_salary_period                                    9         90%
  employer_website                                     5         50%
  job_benefits                                         4         40%
  job_benefits_strings                                 4         40%
  job_city                                             2         20%
  job_state                     

In [10]:
ingested_at = datetime.now(timezone.utc).isoformat()

snowflake_rows = [
    {
        "RAW_PAYLOAD": job,
        "INGESTED_AT": ingested_at,
    }
    for job in jobs
]

print(f"Rows ready for insertion: {len(snowflake_rows)}")
print("\nSample row (key fields only):")
print(json.dumps(
    {
        "RAW_PAYLOAD": {k: v for k, v in snowflake_rows[0]["RAW_PAYLOAD"].items() if k in [
            "job_id", "job_title", "employer_name", "job_location", "job_posted_at"
        ]},
        "INGESTED_AT": snowflake_rows[0]["INGESTED_AT"],
    },
    indent=2
))
print("\n(RAW_PAYLOAD above is truncated to 5 fields — actual row contains all fields)")

Rows ready for insertion: 10

Sample row (key fields only):
{
  "RAW_PAYLOAD": {
    "job_id": "lLMZZm-wci-hYzrpAAAAAA==",
    "job_title": "Data Quality Analyst \u2013 Enterprise Data",
    "employer_name": "Bloomberg",
    "job_posted_at": "1 day ago",
    "job_location": "New York, NY"
  },
  "INGESTED_AT": "2026-05-31T16:37:55.088588+00:00"
}

(RAW_PAYLOAD above is truncated to 5 fields — actual row contains all fields)


In [14]:
# # Cell 11 — Pagination test (costs 1 request)

# # Guard: only run if we don't already have a page-2 cache
# PAGE2_CACHE = CACHE_DIR / "jsearch_page2_response.json"

# if PAGE2_CACHE.exists():
#     print("Page 2 cache already exists — loading from file.")
#     with PAGE2_CACHE.open() as f:
#         page2_data = json.load(f)
# else:
#     # Pull the cursor from the first response
#     cursor = raw_data.get("data", {}).get("cursor")
    
#     if not cursor:
#         raise ValueError("No cursor found in first response — check raw_data['data'] keys.")
    
#     print(f"Cursor found: {cursor[:60]}...")  # truncate for display
    
#     headers = {
#         "x-rapidapi-key":  RAPIDAPI_KEY,
#         "x-rapidapi-host": RAPIDAPI_HOST,
#         "Content-Type":    "application/json",
#     }
    
#     params = {
#         "query":       EXPLORATION_QUERY,
#         "num_pages":   "1",
#         "country":     "us",
#         "date_posted": "all",
#         "cursor":      cursor,   # <-- this is the only change from the first request
#     }
    
#     log.info("Firing page 2 request — 1 more credit consumed.")
    
#     response = requests.get(
#         JSEARCH_BASE_URL,
#         headers=headers,
#         params=params,
#         timeout=30,
#     )
#     response.raise_for_status()
#     page2_data = response.json()
    
#     with PAGE2_CACHE.open("w") as f:
#         json.dump(page2_data, f, indent=2)
    
#     log.info(f"Page 2 saved to {PAGE2_CACHE}")

2026-05-20 22:12:56,047 [INFO] Firing page 2 request — 1 more credit consumed.


Cursor found: EvcDCrcDQU1uMy15VDF5b0J5N0szSGF6QkU4eUFfY2MyR3pCRTZ1S0RhN1FM...


2026-05-20 22:13:00,505 [INFO] Page 2 saved to cache/jsearch_page2_response.json


In [16]:
# # Cell 12 — Verify page 2 is actually different results
# page1_ids = {job["job_id"] for job in raw_data["data"]["jobs"]}
# page2_jobs = page2_data.get("data", {}).get("jobs", [])
# page2_ids  = {job["job_id"] for job in page2_jobs}

# overlap = page1_ids & page2_ids
# print(f"Page 1 job count : {len(page1_ids)}")
# print(f"Page 2 job count : {len(page2_ids)}")
# print(f"Overlapping IDs  : {len(overlap)}")
# print(f"New cursor exists: {'yes' if page2_data.get('data', {}).get('cursor') else 'NO — pagination ends here'}")
# print()

# if overlap:
#     print("⚠️  Overlap detected — cursor pagination may not be working correctly.")
# else:
#     print("✅ No overlap — cursor is advancing correctly.")

# # Fixed Recency check (handles None values safely)
# print("\n--- Recency check (are results newest-first?) ---")
# for job in page2_jobs[:5]:
#     posted_at = job.get('job_posted_at')
#     posted_str = str(posted_at) if posted_at is not None else "NULL"
#     title_str = str(job.get('job_title', ''))[:50]
#     print(f"  {posted_str:<20} {title_str}")

Page 1 job count : 10
Page 2 job count : 10
Overlapping IDs  : 0
New cursor exists: yes

✅ No overlap — cursor is advancing correctly.

--- Recency check (are results newest-first?) ---
  NULL                 Data Analyst, Insurance
  6 days ago           Data Scientist, AVP
  1 day ago            Remote Associate Data Analyst — IVA Insights & UX
  9 days ago           Senior Data Analyst (Data Analytics)
  NULL                 Data Analyst, Customer Intelligence
